# Premium-Seller — a quantitative teardown 🔬
### Total-return spread vs the underlying · the upside/downside capture asymmetry

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Covered-call income beats?: Busted](https://img.shields.io/badge/Covered--call_income_beats%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). QYLD trailed the index it holds because of a punishing capture asymmetry.

> ⚠️ **Not investment advice.** QYLD/QQQ/SPY, monthly total return (Yahoo), 2014–2026. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (premium_seller/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from premium_seller import data, strategy as st
ret = data.fetch_panel()                       # cache-first
al = ret[["QYLD","QQQ","SPY"]].dropna()
cap = st.capture(ret, "QYLD", "QQQ")


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | QYLD−QQQ −10.9%/yr, Sharpe −1.05, t −3.6 |
| Tradability | **Mirage** | Sharpe 0.76 < 1.06; bad asymmetry |
| Income beats? | **Busted** | 50% upside vs 58% downside capture |

> 💡 *In plain words:* the income is gains given back.

## 1 · The claim, steelmanned

- **H₁:** QYLD beats its underlying (QQQ) on total return.
- **H₂:** it does so risk-adjusted (Sharpe/drawdown).
- **H₃:** the capture is favourable (keeps more upside than downside).

## 2 · So what? — what rides on each

If any hold, covered-call income is a real edge. If none do, it's equity beta with the right tail sold off.

## 3 · How we'd know — the protocol

Leg total returns vs QQQ → the spread and its t-stat → the upside/downside capture decomposition.

## 4 · The teardown

### 4.1 The spread vs its own underlying

In [2]:
import pandas as pd
display(pd.DataFrame({'QYLD':st.leg_summary(al,'QYLD'),'QQQ':st.leg_summary(al,'QQQ')}).T[['cagr','sharpe','vol_ann','max_drawdown']].round(3))
print({k:round(v,3) for k,v in st.spread_stats(st.spread(ret,'QYLD','QQQ')).items()})

,cagr,sharpe,vol_ann,max_drawdown
QYLD,0.081,0.764,0.110,-0.225
QQQ,0.192,1.061,0.183,-0.327


{'mean_ann': -0.109, 'sharpe': -1.045, 'tstat': -3.615, 'hit_rate': 0.393, 'n': 150}


> 💡 *In plain words:* −10.9%/yr at t −3.6 vs the index it holds. **H₁, H₂ rejected.**

### 4.2 The capture asymmetry

In [3]:
print({k:round(v,3) for k,v in cap.items()})
print(f"upside capture {cap['upside_capture']:.0%}, downside capture {cap['downside_capture']:.0%}")

{'up_fund': 0.024, 'up_underlying': 0.047, 'down_fund': -0.022, 'down_underlying': -0.038, 'upside_capture': 0.501, 'downside_capture': 0.577}
upside capture 50%, downside capture 58%


> 💡 *In plain words:* keeps 50% of the up, takes 58% of the down — the wrong way round. **H₃ rejected.** Selling calls truncates the right tail far more than the premium buffers the left.

## 5 · The verdict

H₁, H₂, H₃ all rejected → Signal `NONE`, Tradability `MIRAGE`, the income claim `BUSTED`.

## 6 · Could you trade it?

No edge; in a rising market it bleeds relative to the index. The honest version of selling vol is a *sized* short-vol sleeve owning its crash risk ([63 Free-Fall](../../63-free-fall/)), not a buy-and-hold covered-call fund.

## 7 · Going further

Forks: (a) JEPI/JEPQ (active, partial overwrite) vs QYLD (systematic, full overwrite); (b) the Israelov-Nielsen decomposition (equity + short-vol + timing); (c) after-tax (distributions are tax-inefficient). Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).